# Text-to-SQL — Data Prep

Loads [`b-mc2/sql-create-context`](https://huggingface.co/datasets/b-mc2/sql-create-context) (78k schema + question + SQL examples), formats it into chat-style messages, and splits into train/val/test. Run in Colab, CPU only.

In [1]:
!pip install -q -U datasets huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 13.7 MB/s eta 0:00:00


In [2]:
from datasets import load_dataset
import os, shutil

PROJECT_DIR = '/content/text-to-sql-llm'
os.makedirs(f"{PROJECT_DIR}/data", exist_ok=True)

## Load the raw dataset

In [3]:
raw = load_dataset("b-mc2/sql-create-context")
print(raw)
print(raw["train"][0])

README.md:   0%|          | 0.00/4.43k [00:00<?, ?B/s]

sql_create_context_v4.json: reconstructing file:   0%|          |  0.00B / 21.8MB            

sql_create_context_v4.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/78577 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['answer', 'question', 'context'],
        num_rows: 78577
    })
})
{'answer': 'SELECT COUNT(*) FROM head WHERE age > 56', 'question': 'How many heads of the departments are older than 56 ?', 'context': 'CREATE TABLE head (age INTEGER)'}


## Format into chat messages

Matches the Qwen2.5-Instruct chat template used directly by `SFTTrainer` later.

In [4]:
SYSTEM_PROMPT = (
    "You are a precise text-to-SQL assistant. Given a database schema and a "
    "natural language question, output ONLY the SQL query that answers it. "
    "No explanation, no markdown, just the query."
)

def format_example(example):
    user_msg = f"Schema:\n{example['context']}\n\nQuestion: {example['question']}"
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": example["answer"]},
        ]
    }

formatted = raw["train"].map(format_example, remove_columns=raw["train"].column_names)
print(formatted[0]["messages"])

Map:   0%|          | 0/78577 [00:00<?, ? examples/s]

[{'role': 'system', 'content': 'You are a precise text-to-SQL assistant. Given a database schema and a natural language question, output ONLY the SQL query that answers it. No explanation, no markdown, just the query.'}, {'role': 'user', 'content': 'Schema:\nCREATE TABLE head (age INTEGER)\n\nQuestion: How many heads of the departments are older than 56 ?'}, {'role': 'assistant', 'content': 'SELECT COUNT(*) FROM head WHERE age > 56'}]


## Shuffle, subset, and split

90/5/5 train/val/test split on a 10k subset — fast to fine-tune on a free T4.

In [5]:
formatted = formatted.shuffle(seed=42)

SUBSET_SIZE = 10000
subset = formatted.select(range(min(SUBSET_SIZE, len(formatted))))

split_a = subset.train_test_split(test_size=0.1, seed=42)
split_b = split_a["test"].train_test_split(test_size=0.5, seed=42)

train_ds, val_ds, test_ds = split_a["train"], split_b["train"], split_b["test"]
print(f"train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}")

train=9000  val=500  test=500


## Save and download

In [6]:
train_ds.save_to_disk(f"{PROJECT_DIR}/data/train")
val_ds.save_to_disk(f"{PROJECT_DIR}/data/val")
test_ds.save_to_disk(f"{PROJECT_DIR}/data/test")

shutil.make_archive("/content/text-to-sql-data", "zip", f"{PROJECT_DIR}/data")

from google.colab import files
files.download("/content/text-to-sql-data.zip")

Saving the dataset (0/1 shards):   0%|          | 0/9000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/500 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/500 [00:00<?, ? examples/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>